# 02 — Preprocessing & Mineralogie-EDA

Transformation von df_clean → df_transformed, Mineralogie-Testfall charakterisieren.

**Input:**  `data/processed/df_clean.parquet`  
**Output:** `data/processed/df_transformed.parquet`

**Pipeline:** `df_clean → ClassificationTransformer → df_transformed`

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core.classification_transform import ClassificationTransformer
from core.data_explorer import Marc21Explorer, DDC_MAIN
from core.filter_theses import Filter

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

## 1 — df_clean laden

In [ ]:
df_clean = pd.read_parquet(DATA_PROCESSED / "df_clean.parquet")

print(f"Shape: {df_clean.shape}")
print(f"Spalten: {list(df_clean.columns)}")

## 2 — ClassificationTransformer anwenden

In [ ]:
# Nach read_parquet — einmalig alle Listenfelder reparieren
LIST_COLS = ["082_a", "082_2", "083_a", "083_2", "sdnb_codes", "subjects"]
for col in LIST_COLS:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(
            lambda x: list(x) if hasattr(x, "__iter__") and not isinstance(x, str) else (x or [])
        )

In [ ]:
df_transformed = ClassificationTransformer.transform(df_clean)

print("=== Neue Spalten ===")
new_cols = ["ddc_primary_3digit", "has_sdnb", "is_mineralogie"]
print(df_transformed[new_cols].head())

print(f"\nMineralogie-Records: {df_transformed['is_mineralogie'].sum():,}")
print(f"Hat SDNB:            {df_transformed['has_sdnb'].sum():,} ({df_transformed['has_sdnb'].mean()*100:.1f}%)")
print(f"Hat DDC:             {df_transformed['ddc_primary_3digit'].ne('').sum():,}")

## 3 — Plot 4: SDNB-Abdeckung über Zeit

In [ ]:
explorer = Marc21Explorer(df_transformed)

fig = explorer.plot_coverage_by_year(year_min=1924, year_max=2024)
fig.update_layout(title="Klassifikationsabdeckung DNB-Hochschulschriften 1924–2024")
fig.show()

##### Frühe Jahrgänge haben sehr geringe Fallzahlen, DDC-Anteile daher wenig aussagekräftig.

In [ ]:
# Test: Wie viele Records mit DDC haben ein Jahr vor 1950?
df_early_ddc = df_transformed[
    df_transformed["ddc_primary_3digit"].str.len() > 0
]
df_early_ddc = df_early_ddc[
    df_early_ddc["publication_year"].between(1910, 1945)
]

print(f"Records mit DDC vor 1945: {len(df_early_ddc):,}")
print(df_early_ddc[["title", "publication_year", "ddc_primary_3digit"]].head(10).to_string())

## 4 — Plot 5: DDC-Hauptklassenverteilung

In [ ]:
ddc_dist = (
    df_transformed[df_transformed["ddc_primary_3digit"].str.len() > 0]
    .assign(ddc_main=lambda d: d["ddc_primary_3digit"].str[0] + "00")
    .groupby("ddc_main")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=True)
)

# Label anreichern
ddc_dist["label"] = ddc_dist["ddc_main"].apply(
    lambda x: f"{x} — {DDC_MAIN.get(x[0], '?')}"
)

fig = px.bar(
    ddc_dist, x="count", y="label",
    orientation="h",
    title="DDC-Hauptklassenverteilung (Gesamtkorpus)",
    labels={"count": "Anzahl Records", "label": ""},
    color="count",
    color_continuous_scale="Blues",
)
fig.update_layout(
    plot_bgcolor="white", height=480,
    coloraxis_showscale=False,
)
fig.update_xaxes(showgrid=True, gridcolor="#EEEEEE")
fig.show()

## 5 — Sunburst 2: Naturwissenschaften (500er) — Mineralogie hervorheben

In [ ]:
df_hier = explorer.build_ddc_hierarchy()

# Nur 500er Klasse
df_nat = df_hier[df_hier["DDC_1"].str.startswith("Naturwissenschaften")].copy()
df_nat["is_mineralogie"] = df_nat["DDC_3"].str.startswith("549")

fig = px.sunburst(
    df_nat,
    path=["DDC_1", "DDC_2", "DDC_3"],
    values="count",
    title="Naturwissenschaften (500er) — Mineralogie 549 hervorgehoben",
    color="is_mineralogie",
    color_discrete_map={True: "#E53935", False: "#90CAF9"},
)
fig.update_traces(
    hovertemplate="<b>%{label}</b><br>Records: %{value:,}<extra></extra>",
)
fig.update_layout(height=580)
fig.show()

In [ ]:
# Test: Records mit DDC 549 aus df_transformed
# df_549 = df_transformed[df_transformed["ddc_primary_3digit"].str.startswith("549", na=False)].copy()

#print(f"Records: {len(df_549):,}")
#print(df_549[["title", "publication_year", "ddc_primary_3digit"]].to_string())
#print(df_549["ddc_primary_3digit"].value_counts())


## 6 — Mineralogie-Testfall charakterisieren

In [ ]:
df_min = df_transformed[df_transformed["is_mineralogie"]].copy()

print(f"=== Mineralogie-Records ===")
print(f"Gesamt:           {len(df_min):,}")
print(f"Via DDC 549:      {df_min['ddc_primary_3digit'].str.startswith('549').sum():,}")
print(f"Via SDNB 38:      {df_min['has_sdnb'].sum():,}")

df_min["decade"] = (df_min["publication_year"] // 10 * 10).astype("Int16")
print("\nNach Jahrzehnt:")
print(df_min.groupby("decade").size().to_string())

In [ ]:
fig = explorer.plot_mineralogie_by_decade()
fig.show()

### Retro-Bedarf: Unklassifizierte Mineralogie-Records nach Jahrzehnt

In [ ]:
fig = explorer.plot_retro_bedarf()
fig.show()

### Retro-Kandidaten identifizieren

In [ ]:
retro = (
    Filter(df_transformed)
    .filter_by_year(year_range=(1940, 1979))
    .filter_mineralogie()
    .filter_unclassified()
    .result()
)

print(f"Retro-Kandidaten (1940–1979, unklassifiziert): {len(retro):,}")
print(retro[["title", "publication_year"]].head(10).to_string())

## 7 — df_transformed speichern

In [ ]:
out_path = DATA_PROCESSED / "df_transformed.parquet"
df_transformed.to_parquet(out_path, engine="pyarrow", compression="snappy")
print(f"Gespeichert: {out_path}")
print(f"Größe:       {out_path.stat().st_size / 1024**2:.1f} MB")